In [17]:
import gradio as gr

# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from IPython.display import Image
from io import BytesIO
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI
import ollama

import requests
from bs4 import BeautifulSoup

import pandas as pd
import ast   # or json if it's valid JSON
from urllib.parse import urljoin

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata

In [18]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [19]:
def system_prompt():
    system_prompt = {
        "Improved Version": (
            "You are an adaptive learner who retains all read information. "
            "Your primary task is to fetch and analyze content from a provided content already fatched from url. "
            "When a user asks a question, provide a text response under 50 words "
            "based strictly on the URL data. Note that content can be any topic "
            "either person, animal, history, geography, Science, Engineering, Medical etc..."
        ),
        "File Saving Rules": {
            "Name format": "Save content as <Subject_Name>.txt",
            "Topic matching": "Determine the filename from the URL topic",
            "Handling duplicates": "Append sequential suffixes (_01, _02) if files exist"
        },
        "Out-of-Scope Queries": {
            "Rule": "Reject questions unrelated to the content",
            "Response": "This question is out of bounds. Please only ask about [Subject Name]."
        },
        "Example Scenario": {
            "content": "info about Albert Einstein",
            "File Name": "Albert_Einstein.txt",
            "Saved Data": "Track his achievements, family, and famous works.",
            "Unrelated Query": "What color is the dog?",
            "Required Response": "This question is out of bounds. Please only ask about Albert Einstein."
        }
    }
    return system_prompt


In [20]:
def get_user_prompt(url: str):
    # Call your custom API to fetch contents
    contents = fetch_website_contents(url=url)

    user_prompt = {
        "User Prompt Template": {
            "Content to Analyze": contents,
            "Instructions": (
                "Analyze the given content. "
                "Save the text under a relevant filename using the <Subject_Name>.txt format. "
                "Memorize the key details regarding the subject. "
                "Stand by to answer my questions using responses under 50 words. "
                "Remind me if my questions go out of bounds."
            )
        }
    }
    return user_prompt

## Get Models supported by System

In [21]:
def list_openai_models():
    """Fetch available OpenAI models from your account."""
    client = OpenAI()
    models = client.models.list()
    return [m.id for m in models.data]   # return as list

def list_ollama_models():
    models = ollama.list()
    # Case 1: if dicts with "model" key
    return [m["model"] for m in models["models"]]

# Combine both lists into one
all_models = list_openai_models() + list_ollama_models()

In [22]:
def get_your_answer(select_model: str, url: str, user_questions: str):
    """
    Wrapper function to query either OpenAI or Ollama depending on model selection.
    Validates model availability before running.
    """
    # Build context
    context = redacted_contents(url=url,headers=headers)   
    SIMPLE_PROMPT = [
        {"role": "system", "content": "Answer only from the provided context. If it's not there, say you don't know."},
        {"role": "user", "content": f"\n{context}\n\n\nQuestion: {user_questions}"},
    ]

    # Check Ollama models
    available_ollama = list_ollama_models()
    if select_model in available_ollama:
        response = ollama.chat(
            model=select_model,
            messages=SIMPLE_PROMPT
        )
        return response["message"]["content"]

    # Check OpenAI models
    available_openai = list_openai_models()
    if select_model in available_openai:
        client = OpenAI()
        response = client.chat.completions.create(
            model=select_model,
            messages=SIMPLE_PROMPT
        )
        return response.choices[0].message.content

    # If model not found in either list
    raise ValueError(f"Model '{select_model}' is not available on your system or OpenAI account.")


In [23]:
def get_your_answer_another(select_model: str, url: str, user_questions: str):
    """
    Wrapper function to query either OpenAI or Ollama depending on model selection.
    Validates model availability before running.
    """
    # Build context
    context = redacted_contents(url=url, headers=headers)
    if isinstance(context, list):
        context = "\n".join(map(str, context))  # normalize to string

    SIMPLE_PROMPT = [
        {"role": "system", "content": "Answer only from the provided context. If it's not there, say you don't know."},
        {"role": "user", "content": f"\n{context}\n\n\nQuestion: {user_questions}"},
    ]

    # Check Ollama models
    available_ollama = list_ollama_models()
    if select_model in available_ollama:
        stream = ollama.chat(model=select_model, messages=SIMPLE_PROMPT, stream=True)
        output = ""
        for chunk in stream:
            if "message" in chunk and "content" in chunk["message"]:
                output += chunk["message"]["content"]
        return output

    # Check OpenAI models
    available_openai = list_openai_models()
    if select_model in available_openai:
        client = OpenAI()
        stream = client.chat.completions.create(model=select_model, messages=SIMPLE_PROMPT, stream=True)
        output = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
                output += chunk.choices[0].delta.content
        return output

    # If model not found in either list
    raise ValueError(f"Model '{select_model}' is not available on your system or OpenAI account.")


In [ ]:
qus_input = gr.Textbox(
    label="Your questions:", 
    info="Enter a qus for the LLM", 
    lines=7
)

model_selector = gr.Dropdown(
    choices=all_models,      # dynamically populated
    label="Select model",
    value=all_models[0] if all_models else None   # default to first if available
)

user_url = gr.Textbox(
    label="Your URL:", 
    info="Enter a URL for analysis", 
    lines=1
)

message_output = gr.Markdown(label="Response:")

def stream_model(qus_input, model_selector,user_url):
    print(f"{user_url}")
    return get_your_answer_another(select_model = model_selector, 
                           url = user_url,
                           user_questions = qus_input)

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[qus_input, model_selector,user_url],     
    outputs=[message_output],     
    flagging_mode="never",    
)
view.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


https://en.wikipedia.org/wiki/Albert_Einstein
https://en.wikipedia.org/wiki/Albert_Einstein
https://en.wikipedia.org/wiki/Albert_Einstein
https://en.wikipedia.org/wiki/Albert_Einstein
